In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch
import pandas as pd
import os

In [3]:
os.chdir("/content/drive/MyDrive/transformers/Ecommerce/CustomerFeedbackAnalysis")

In [4]:
! ls

01_data_collection_and_balancing.ipynb.ipynb  ProductsReviews.zip
balanced_dataset.csv			      results
FinalModel				      training_model.ipynb
images					      Use_Trained_Model.ipynb
ProductsReviews.csv


In [ ]:
trained_model = RobertaForSequenceClassification.from_pretrained("FinalModel")

In [6]:
tokenizer = RobertaTokenizer.from_pretrained("./FinalModel")

In [11]:
df = pd.read_csv("ProductsReviews.csv")

In [12]:
df_reviews = df[["reviews.text", "sentiment"]]

In [13]:
df_reviews

,reviews.text,sentiment
0,Purchased on Black FridayPros - Great Price (e...,Positive
1,I purchased two Amazon in Echo Plus and two do...,Positive
2,Just an average Alexa option. Does show a few ...,Neutral
3,"very good product. Exactly what I wanted, and ...",Positive
4,This is the 3rd one I've purchased. I've bough...,Positive
...,...,...
3995,"It‚Äôs fun for the family to play with, but it...",Positive
3996,"I love the Kindle, it is a great product. It r...",Positive
3997,I was looking for a blutooth speaker to use wi...,Positive
3998,This is the second Amazon Fire 7 tablet I have...,Positive


In [14]:
df_reviews["sentiment"].unique()

array(['Positive', 'Neutral', 'Negative'], dtype=object)

In [15]:
def map_sentiment_to_class(sentiment):
  if sentiment == "Positive":
    return 2
  if sentiment == "Neutral":
    return 1
  return 0

In [ ]:
df_reviews["class"] = df_reviews["sentiment"].apply(map_sentiment_to_class)

In [18]:
df_reviews

,reviews.text,sentiment,class
0,Purchased on Black FridayPros - Great Price (e...,Positive,2
1,I purchased two Amazon in Echo Plus and two do...,Positive,2
2,Just an average Alexa option. Does show a few ...,Neutral,1
3,"very good product. Exactly what I wanted, and ...",Positive,2
4,This is the 3rd one I've purchased. I've bough...,Positive,2
...,...,...,...
3995,"It‚Äôs fun for the family to play with, but it...",Positive,2
3996,"I love the Kindle, it is a great product. It r...",Positive,2
3997,I was looking for a blutooth speaker to use wi...,Positive,2
3998,This is the second Amazon Fire 7 tablet I have...,Positive,2


### Define an a function to get the accuracy of the model

In [19]:
def get_model_accuracy(model, tokenizer, dframe):
  df = dframe.copy()
  df["predicted_class"] = 0
  for index, row in df.iterrows():
    sentiment = row["reviews.text"]
    tokenized_sentiment = tokenizer(sentiment, return_tensors="pt", truncation=True, padding= True, max_length= 512)
    output= model(**tokenized_sentiment)
    logits= output.logits
    model_prediction_value = torch.argmax(logits, dim=1).item()
    df.at[index, "predicted_class"] = model_prediction_value
  correct_predictions = (df['predicted_class']  == df['class']).sum()
  accuracy = round((correct_predictions/len(df)) * 100, 2 )

  return accuracy




In [20]:
df_sample = df_reviews.sample(frac=.1)

In [21]:
acc = get_model_accuracy(trained_model, tokenizer, df_sample)
print(acc)

74.25


### Define a function for sentence classification

In [22]:
def predict_sentiment(text, model, tokenizer):
  tokenized_sentiment = tokenizer(text, return_tensors="pt", truncation=True, padding= True, max_length= 512)
  output= model(**tokenized_sentiment)
  logits= output.logits
  model_prediction_value = torch.argmax(logits, dim=1).item()

  if model_prediction_value == 2:
    return "Positive"
  elif model_prediction_value == 1:
    return "Neutral"
  return "Negative"


In [26]:
predict_sentiment("I absolutely love this product, the quality is excellent and it works perfectly.", model=trained_model, tokenizer= tokenizer)

'Positive'

In [24]:
predict_sentiment("The product works as expected, nothing particularly special.", model=trained_model, tokenizer= tokenizer)

'Neutral'

In [25]:
predict_sentiment("I am disappointed with this product, it broke after a few uses", model=trained_model, tokenizer= tokenizer)

'Negative'

##  Model: Twitter RoBERTa for Sentiment Analysis

We use **`twitter-roberta-base-sentiment-latest`**, a RoBERTa model trained on ~124M tweets and fine-tuned for sentiment analysis.  
It classifies text into **negative, neutral, and positive** sentiments.  
This model is optimized for **English social media text**.

In [27]:
model_name = f"cardiffnlp/twitter-roberta-base-sentiment-latest"

In [ ]:
twitter_sentiment_model = RobertaForSequenceClassification.from_pretrained(model_name)

In [30]:
print(predict_sentiment("I absolutely love this product, the quality is excellent and it works perfectly.", model=twitter_sentiment_model, tokenizer= tokenizer))
print(predict_sentiment("The product works as expected, nothing particularly special.", model=twitter_sentiment_model, tokenizer= tokenizer))
print(predict_sentiment("I am disappointed with this product, it broke after a few uses", model=twitter_sentiment_model, tokenizer= tokenizer))

Positive
Neutral
Negative


In [33]:
twitter_sentiment_model_accurracy = get_model_accuracy(twitter_sentiment_model, tokenizer, df_sample)
print(twitter_sentiment_model_accurracy)

88.25


In [34]:
twitter_sentiment_model_accurracy = get_model_accuracy(twitter_sentiment_model, tokenizer, df_reviews)
print(twitter_sentiment_model_accurracy)

89.55


In [37]:
print(f"Accuracy of Twitter Sentiment Model: {twitter_sentiment_model_accurracy} %")
print(f"Accuracy of Our Trained Model: {acc} %")

Accuracy of Twitter Sentiment Model: 89.55 %
Accuracy of Our Trained Model: 74.25 %
